In [0]:
from datetime import datetime
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "finalproject")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')

## Creamos el parámetro con la fecha de carga prueba

In [0]:
dbutils.widgets.text("fecha_carga", datetime.today().strftime("%Y-%m-%d"), "Fecha de carga")

In [0]:
fecha_carga = datetime.strptime(dbutils.widgets.get('fecha_carga')[:10], "%Y-%m-%d").date()

## Transformamos la tabla bronze compras

In [0]:
maxima_fecha = spark.sql(f"SELECT max(max_fecha) FROM {catalog_name}.bronze.watermarks").first()[0]

In [0]:
df_compras = spark.sql(f"""
    SELECT 
        ventaid, factura, fecha_orden, fecha_entrega, fecha_envio,
        estado, cliente_code, tipo_cliente, nombres, apellidos,
        vendedor, departamento, metodo_pago, tipo_compra, created_at, updated_at
    FROM {catalog_name}.bronze.compras where updated_at > '{maxima_fecha}'
                       """)

In [0]:
df_compras = df_compras.withColumns(
    {
        'ventaid' : col('ventaid').cast('integer'),
        'estado' : col('estado').cast('integer'),
        'fecha_orden' : to_date(col('fecha_orden'), 'dd/MM/yyyy'),
        'fecha_entrega' : to_date(col('fecha_entrega'), 'dd/MM/yyyy'),
        'fecha_envio' : to_date(trim(col('fecha_envio')), 'dd/MM/yyyy'),
        'factura' : upper(trim(col('factura'))),
        'tipo_cliente' : upper(trim(col('tipo_cliente'))),
        'nombres' : initcap(trim(col('nombres'))),
        'apellidos' : initcap(trim(col('apellidos'))),
        'vendedor' : trim(col('vendedor')),
        'departamento' : trim(col('departamento')),
        'metodo_pago' : trim(col('metodo_pago'))
    }
)

In [0]:
df_compras = df_compras\
  .withColumns(
    {
      'estado': when(col('estado') == 1,'Creado')\
                .when(col('estado') == 2,'En Curso')\
                .when(col('estado') == 3,'Programado')\
                .when(col('estado') == 4,'Cancelado')\
                .when(col('estado') == 5,'Entregado')\
                .otherwise('No definido'),
      'cliente_id': split(col('cliente_code'),' - ')[0].cast('integer'),
      'num_documento': split(col('cliente_code'),' - ')[1],
    }
  ).withColumn('num_documento', lpad(col('num_documento'), 8, '0'))

In [0]:
df_compras = df_compras\
.withColumn(
    'tipo_documento', when(length(col('num_documento')) == 8, 'DNI')\
                        .when((length(col('num_documento')) == 11) & (col('num_documento').startswith('10')) , 'RUC10')\
                        .when((length(col('num_documento')) == 11) & (col('num_documento').startswith('20')) , 'RUC20')\
                        .otherwise(None)
).withColumn(
    'nombre_cliente', concat_ws(' ', col('nombres'), col('apellidos'))
)

In [0]:
df_compras = df_compras.withColumns({
    'fecha_envio': when(col('estado').isin(['Creado','En Curso']), lit(None).cast('date'))\
                   .otherwise(col('fecha_envio')),
    
    # Si el estado NO es 5 (Entregado), entonces no hay fecha de entrega
    'fecha_entrega': when(~col('estado').isin(['Cancelado','Entregado']), lit(None).cast('date'))\
                       .otherwise(col('fecha_entrega')),
})

In [0]:
df_compras = df_compras\
    .withColumn(
        'dias_abierto',when(
                    col('estado').isin(['Creado', 'En Curso', 'Programado']),
                    datediff(lit(fecha_carga), col('fecha_orden'))
                ).otherwise(datediff(col('fecha_entrega'), col('fecha_orden')))
    ).withColumn(
        'grupo_dias_abierto',when(col('dias_abierto').isNull(),None)\
                            .when(col('dias_abierto') <= 3, '[0 - 3 días]')\
                            .when(col('dias_abierto') <= 7, '[4 - 7 días]')\
                            .when(col('dias_abierto') >= 8, '[más de 8 días]')
    )

In [0]:
df_compras = df_compras.select('ventaid','factura','tipo_compra','fecha_orden','fecha_entrega','fecha_envio','estado','cliente_id','tipo_documento','num_documento','nombre_cliente','tipo_cliente','vendedor','departamento','metodo_pago','dias_abierto','grupo_dias_abierto','created_at','updated_at').withColumn('fecha_carga', current_timestamp())

## Transformamos la tabla bronze detalles

In [0]:
maxima_fecha_detalle = spark.sql(f"""select coalesce(max(fecha_carga),'1900-01-01') from {catalog_name}.bronze.detalles""").first()[0]

In [0]:
df_detalles = spark.sql(f"""
    SELECT 
        detalle_id, factura, oferta_id, categoria, subcategoria,
        producto, unidades, precio_unitario, tienda, fecha_carga
    FROM {catalog_name}.bronze.detalles WHERE fecha_carga > '{maxima_fecha_detalle}'
                       """)

In [0]:
df_detalles = df_detalles.withColumns({
    'unidades' : col('unidades').cast('integer'),
    'precio_unitario' : col('precio_unitario').cast('double')
})

In [0]:
df_detalles = df_detalles.withColumns({
    'detalle_id' : col('detalle_id').cast('integer'),
    'unidades' : col('unidades').cast('integer'),
    'oferta_id' : col('oferta_id').cast('integer'),
    'precio_unitario' : col('precio_unitario').cast('double'),
    'factura' : upper(trim(col('factura'))),
    'categoria': trim(col('categoria')),
    'subcategoria': trim(col('subcategoria')),
    'producto': trim(col('producto'))
}).withColumn('subtotal',col('unidades') * col('precio_unitario')).select(
    'detalle_id','factura','tienda','oferta_id','categoria','subcategoria','producto','unidades','subtotal'
).withColumn('fecha_carga', current_timestamp())

In [0]:
try:
    if not spark.catalog.tableExists(f'{catalog_name}.silver.detalles'):
        df_detalles.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.silver.detalles')
    else:
        detalles = DeltaTable.forName(spark,f'{catalog_name}.silver.detalles')
        detalles.alias('hist')\
        .merge(
            df_detalles.alias('inc'),
            'hist.detalle_id = inc.detalle_id'
        ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

    if not spark.catalog.tableExists(f'{catalog_name}.silver.compras'):
        df_compras.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable(f'{catalog_name}.silver.compras')
    else:
        compras = DeltaTable.forName(spark,f'{catalog_name}.silver.compras')
        compras.alias('hist')\
        .merge(
            df_compras.alias('inc'),
            'hist.detalle_id = inc.detalle_id'
        ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
        
except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error 
    error_summary = str(e)
    # Traza del error (ver en qué parte se generó el error)
    error_trace = traceback.format_exc()

    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full
    
    dbutils.jobs.taskValues.set(key = "error", value = error_msg)
    raise e